In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev",
    choices=["fq_dev", "fq_test", "fq_prod"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="NETSUITE",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="fact_financial_pnl",
    choices=["discount", "sales", "fact_financial_pnl", "fact_financial_pnl"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]

print(f"Environment: {environment}")
print(f"Source: {source}")
print(f"Domain: {domain}")

In [0]:
%sql
select * from fq_dev_catalog.bronze.gl_report limit 1

In [0]:
pwd

In [0]:
path = '/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git'

In [0]:
import os
os.environ['PYTHONPATH'] = f"{path}/src"

# import sys
# sys.path.append(f'{path}/src')

In [0]:
from pyspark.sql.functions import *
import re

from foodquest_pnl import (
    create_total_row, 
    create_calculated_metric, 
    add_previous_year_data,
    add_net_sales_calculations,
    get_dimension_columns
)

def to_snake_case(name):
    return re.sub(r'[\s\-]+', '_', name).lower()

def to_snake_case_df(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, to_snake_case(col_name))
    return df


def enrich_df(df):
    # Load master tables
    df_coa_master = spark.read.table("fq_dev_catalog.silver.dim_coa_master")
    df_location_master = spark.read.table("fq_dev_catalog.silver.dim_location_master")
    
    dimension_cols = get_dimension_columns()

    from pyspark.sql.functions import col

    join_configs = [
        {
            'df': df_coa_master,
            'left_key': col("accountNumber").cast("string"),
            'right_key': df_coa_master["account_number"].cast("string"),
            'join_type': 'inner'
        },
        {
            'df': df_location_master,
            'left_key': col("location"),
            'right_key': df_location_master["netsuite_location_name"],
            'join_type': 'left'
        }
    ]

    df_all_masters = join_dataframes(df, join_configs)
    
    # Step 1: Join both master tables
    df_all_masters = df.join(
        df_coa_master, 
        df_coa_master["account_number"].cast("string") == df["accountNumber"], 
        'inner'
    ).join(
        df_location_master,
        col("location") == df_location_master.netsuite_location_name,
        'left'
    )
    
    # Step 2: Create detail rows
    df_detail = df_all_masters.select(
        col("location"), col("month"), col("year"),
        col("account_name"), col("name"), col("accoun_type"),
        col("majour_group"), col("group"), col("sub_group"),
        col("amount"),
        *[col(c) for c in dimension_cols],
        lit("Detail").alias("Detail/Total")
    )
    
    # Step 3: Create total rows at different aggregation levels
    df_subgroup_totals = create_total_row(
        df_all_masters,
        ["location", "month", "year", "majour_group", "group", "sub_group"] + dimension_cols,
        "sub_group",
        "Total",
        dimension_cols
    )
    
    df_group_totals = create_total_row(
        df_all_masters,
        ["location", "month", "year", "majour_group", "group"] + dimension_cols,
        "group",
        "Total",
        dimension_cols
    )
    
    df_majour_group_totals = create_total_row(
        df_all_masters,
        ["location", "month", "year", "majour_group"] + dimension_cols,
        "majour_group",
        "Grand Total",
        dimension_cols
    )
    
    # Step 4: Create calculated metrics
    df_gross_profit = create_calculated_metric(
        df_all_masters,
        "Gross Profit",
        abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
        abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))),
        dimension_cols
    )
    
    df_operating_profit = create_calculated_metric(
        df_all_masters,
        "Operating Profit",
        abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
        abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
        abs(sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0))),
        dimension_cols
    )
    
    df_ebitda = create_calculated_metric(
        df_all_masters,
        "EBITDA",
        (abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
         abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
         abs(sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0)))) +
        abs(sum(when(col("majour_group") == "Depreciation & Amortization", col("amount")).otherwise(0))),
        dimension_cols
    )
    
    df_net_profit = create_calculated_metric(
        df_all_masters,
        "Net Profit/(Loss)",
        (abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
         abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
         abs(sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0)))) -
        abs(sum(when(col("majour_group") == "Finance Costs", col("amount")).otherwise(0))) -
        abs(sum(when(col("majour_group") == "Tax", col("amount")).otherwise(0))),
        dimension_cols
    )
    
    # Step 5: Union all dataframes
    df_combined = (df_detail
        .unionAll(df_subgroup_totals)
        .unionAll(df_group_totals)
        .unionAll(df_majour_group_totals)
        .unionAll(df_gross_profit)
        .unionAll(df_operating_profit)
        .unionAll(df_ebitda)
        .unionAll(df_net_profit))
    
    # Step 6: Add previous year data
    df_with_py = add_previous_year_data(df_combined)
    
    # Step 7: Calculate net sales metrics
    df_with_calculations = add_net_sales_calculations(df_with_py)
    
    # Step 8: Select final columns and convert to snake case
    df_final = df_with_calculations.select(
        col("account_name"), col("name"), col("accoun_type"),
        col("majour_group"), col("group"), col("sub_group"),
        *[col(c) for c in dimension_cols],
        col("Detail/Total"), col("amount"), col("year"), col("month"),
        col("actual_net_sales"), col("py_net_sales"),
        col("brand_act_net_sales"), col("brand_py_net_sales"),
        col("company_act_net_sales"), col("company_py_net_sales")
    )
    
    df_final = to_snake_case_df(df_final)
    
    # Step 9: Join with sort order and final selection
    df_sort = df_coa_master.select("account_name", "sort_order", "calculation_type")
    df_final_sort = df_final.join(
        df_sort, 
        df_sort["account_name"].cast("string") == df_final.account_name,
        'inner'
    ).drop(df_sort["account_name"]).select(
        col("account_name"), col("name"),
        col("majour_group"), col("group"), col("sub_group"), col("accoun_type"),
        col("netsuite_location_name"), col("type"), col("location_id"),
        col("brand_id"), col("company_id"), col("store_type"),
        col("parent_company"), col("country_code"), col('zone'), col("city"),
        col("amount"), col("year"), col("month"),
        col("actual_net_sales"), col("py_net_sales"),
        col("brand_act_net_sales"), col("brand_py_net_sales"),
        col("company_act_net_sales"), col("company_py_net_sales"),
        col("detail/total"), col('sort_order'), col('calculation_type')
    )
    
    return df_final_sort.orderBy(
        "parent_company", "company_id", "brand_id", 
        "netsuite_location_name", 'year', 'month', 'sort_order'
    )

In [0]:
# Create the test file
test_code = f"""
import sys
sys.path.append('{path}/src')

import pytest
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, sum as spark_sum, when
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from foodquest_pnl import (
    create_total_row,
    create_calculated_metric,
    add_previous_year_data,
    add_net_sales_calculations,
    get_dimension_columns
)

@pytest.fixture(scope="session")
def spark():
    return SparkSession.builder.getOrCreate()

@pytest.fixture
def sample_data(spark):
    \"\"\"Create sample test data.\"\"\"
    schema = StructType([
        StructField("location", StringType(), True),
        StructField("month", IntegerType(), True),
        StructField("year", StringType(), True),
        StructField("account_name", StringType(), True),
        StructField("name", StringType(), True),
        StructField("accoun_type", StringType(), True),
        StructField("majour_group", StringType(), True),
        StructField("group", StringType(), True),
        StructField("sub_group", StringType(), True),
        StructField("amount", DoubleType(), True),
        StructField("netsuite_location_name", StringType(), True),
        StructField("type", StringType(), True),
        StructField("location_id", StringType(), True),
        StructField("brand_id", StringType(), True),
        StructField("company_id", StringType(), True),
        StructField("parent_company", StringType(), True),
        StructField("country_code", StringType(), True),
        StructField("zone", StringType(), True),
        StructField("store_type", StringType(), True),
        StructField("city", StringType(), True)
    ])
    
    data = [
        ("LOC001", 1, "2024", "Revenue", "Sales Revenue", "Income", "Sales", "Product Sales", "Retail Sales", 10000.0,
         "Store 1", "Retail", "L001", "B001", "C001", "Parent Corp", "US", "East", "Flagship", "NYC"),
        ("LOC001", 1, "2024", "COGS", "Cost of Goods", "Expense", "Purchases", "Product Cost", "Direct Cost", 4000.0,
         "Store 1", "Retail", "L001", "B001", "C001", "Parent Corp", "US", "East", "Flagship", "NYC"),
        ("LOC001", 1, "2024", "Rent", "Store Rent", "Expense", "Overheads", "Fixed Costs", "Occupancy", 2000.0,
         "Store 1", "Retail", "L001", "B001", "C001", "Parent Corp", "US", "East", "Flagship", "NYC"),
    ]
    
    return spark.createDataFrame(data, schema)

@pytest.fixture
def dimension_cols():
    \"\"\"Return dimension columns for testing.\"\"\"
    return get_dimension_columns()


class TestGetDimensionColumns:
    def test_returns_list(self):
        result = get_dimension_columns()
        assert isinstance(result, list)
    
    def test_contains_expected_columns(self):
        result = get_dimension_columns()
        expected = ["netsuite_location_name", "type", "location_id", "brand_id", "company_id"]
        for col in expected:
            assert col in result


class TestCreateTotalRow:
    def test_creates_subgroup_total(self, sample_data, dimension_cols):
        result = create_total_row(
            sample_data,
            ["location", "month", "year", "majour_group", "group", "sub_group"] + dimension_cols,
            "sub_group",
            "Total",
            dimension_cols
        )
        
        # Check schema
        assert "account_name" in result.columns
        assert "amount" in result.columns
        assert "Detail/Total" in result.columns
        
        # Check data
        rows = result.collect()
        assert len(rows) > 0
        assert rows[0]["Detail/Total"] == "Total"
        assert "Total" in rows[0]["account_name"]
    
    def test_aggregates_amounts(self, sample_data, dimension_cols):
        result = create_total_row(
            sample_data,
            ["location", "month", "year", "majour_group"] + dimension_cols,
            "majour_group",
            "Grand Total",
            dimension_cols
        )
        
        rows = result.collect()
        # Should aggregate amounts by major group
        assert len(rows) == 3  # Sales, Purchases, Overheads
    
    def test_null_fields_for_hierarchy(self, sample_data, dimension_cols):
        result = create_total_row(
            sample_data,
            ["location", "month", "year"] + dimension_cols,
            "majour_group",
            "Grand Total",
            dimension_cols
        )
        
        row = result.first()
        assert row["majour_group"] is None
        assert row["group"] is None
        assert row["sub_group"] is None


class TestCreateCalculatedMetric:
    def test_creates_gross_profit(self, sample_data, dimension_cols):
        calculation = (
            spark_sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) -
            spark_sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))
        )
        
        result = create_calculated_metric(
            sample_data,
            "Gross Profit",
            calculation,
            dimension_cols
        )
        
        rows = result.collect()
        assert len(rows) == 1
        assert rows[0]["account_name"] == "Gross Profit"
        assert rows[0]["name"] == "Gross Profit"
        assert rows[0]["amount"] == 6000.0  # 10000 - 4000
    
    def test_metric_has_grand_total_label(self, sample_data, dimension_cols):
        calculation = spark_sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))
        
        result = create_calculated_metric(
            sample_data,
            "Total Sales",
            calculation,
            dimension_cols
        )
        
        row = result.first()
        assert row["Detail/Total"] == "Grand Total"
    
    def test_preserves_dimensions(self, sample_data, dimension_cols):
        calculation = spark_sum(col("amount"))
        
        result = create_calculated_metric(
            sample_data,
            "Test Metric",
            calculation,
            dimension_cols
        )
        
        row = result.first()
        assert row["brand_id"] == "B001"
        assert row["company_id"] == "C001"
        assert row["location_id"] == "L001"


class TestAddPreviousYearData:
    def test_adds_py_amount_column(self, spark):
        schema = StructType([
            StructField("location", StringType(), True),
            StructField("month", IntegerType(), True),
            StructField("year", StringType(), True),
            StructField("account_name", StringType(), True),
            StructField("amount", DoubleType(), True),
            StructField("Detail/Total", StringType(), True)
        ])
        
        data = [
            ("LOC001", 1, "2023-01-01", "Sales", 10000.0, "Detail"),
            ("LOC001", 1, "2024-01-01", "Sales", 12000.0, "Detail"),
        ]
        
        df = spark.createDataFrame(data, schema)
        result = add_previous_year_data(df)
        
        assert "py_amount" in result.columns
    
    def test_matches_py_data(self, spark):
        schema = StructType([
            StructField("location", StringType(), True),
            StructField("month", IntegerType(), True),
            StructField("year", StringType(), True),
            StructField("account_name", StringType(), True),
            StructField("amount", DoubleType(), True),
            StructField("Detail/Total", StringType(), True)
        ])
        
        data = [
            ("LOC001", 1, "2023-01-01", "Sales", 10000.0, "Detail"),
            ("LOC001", 1, "2024-01-01", "Sales", 12000.0, "Detail"),
        ]
        
        df = spark.createDataFrame(data, schema)
        result = add_previous_year_data(df)
        
        # 2024 row should have 2023 amount as py_amount
        row_2024 = result.filter(col("year") == 2024).first()
        assert row_2024["py_amount"] == 10000.0
    
    def test_handles_missing_py_data(self, spark):
        schema = StructType([
            StructField("location", StringType(), True),
            StructField("month", IntegerType(), True),
            StructField("year", StringType(), True),
            StructField("account_name", StringType(), True),
            StructField("amount", DoubleType(), True),
            StructField("Detail/Total", StringType(), True)
        ])
        
        data = [
            ("LOC001", 1, "2024-01-01", "Sales", 12000.0, "Detail"),
        ]
        
        df = spark.createDataFrame(data, schema)
        result = add_previous_year_data(df)
        
        row = result.first()
        assert row["py_amount"] == 0.0  # Should default to 0


class TestAddNetSalesCalculations:
    def test_adds_net_sales_columns(self, spark):
        schema = StructType([
            StructField("location", StringType(), True),
            StructField("month", IntegerType(), True),
            StructField("year", IntegerType(), True),
            StructField("account_name", StringType(), True),
            StructField("amount", DoubleType(), True),
            StructField("py_amount", DoubleType(), True),
            StructField("brand_id", StringType(), True),
            StructField("company_id", StringType(), True)
        ])
        
        data = [
            ("LOC001", 1, 2024, "Total Sales", 10000.0, 9000.0, "B001", "C001"),
        ]
        
        df = spark.createDataFrame(data, schema)
        result = add_net_sales_calculations(df)
        
        expected_cols = [
            "actual_net_sales", "py_net_sales",
            "brand_act_net_sales", "brand_py_net_sales",
            "company_act_net_sales", "company_py_net_sales"
        ]
        
        for col_name in expected_cols:
            assert col_name in result.columns
    
    def test_calculates_location_sales(self, spark):
        schema = StructType([
            StructField("location", StringType(), True),
            StructField("month", IntegerType(), True),
            StructField("year", IntegerType(), True),
            StructField("account_name", StringType(), True),
            StructField("amount", DoubleType(), True),
            StructField("py_amount", DoubleType(), True),
            StructField("brand_id", StringType(), True),
            StructField("company_id", StringType(), True)
        ])
        
        data = [
            ("LOC001", 1, 2024, "Total Sales", 10000.0, 9000.0, "B001", "C001"),
            ("LOC001", 1, 2024, "Other", 5000.0, 4000.0, "B001", "C001"),
        ]
        
        df = spark.createDataFrame(data, schema)
        result = add_net_sales_calculations(df)
        
        rows = result.collect()
        # Both rows should have same actual_net_sales (10000)
        assert rows[0]["actual_net_sales"] == 10000.0
        assert rows[1]["actual_net_sales"] == 10000.0
        assert rows[0]["py_net_sales"] == 9000.0
    
    def test_calculates_brand_sales(self, spark):
        schema = StructType([
            StructField("location", StringType(), True),
            StructField("month", IntegerType(), True),
            StructField("year", IntegerType(), True),
            StructField("account_name", StringType(), True),
            StructField("amount", DoubleType(), True),
            StructField("py_amount", DoubleType(), True),
            StructField("brand_id", StringType(), True),
            StructField("company_id", StringType(), True)
        ])
        
        data = [
            ("LOC001", 1, 2024, "Total Sales", 10000.0, 9000.0, "B001", "C001"),
            ("LOC002", 1, 2024, "Total Sales", 8000.0, 7000.0, "B001", "C001"),
        ]
        
        df = spark.createDataFrame(data, schema)
        result = add_net_sales_calculations(df)
        
        rows = result.collect()
        # Both locations of same brand should have aggregated brand sales
        assert rows[0]["brand_act_net_sales"] == 18000.0
        assert rows[1]["brand_act_net_sales"] == 18000.0


# Run tests
if __name__ == "__main__":
    pytest.main([__file__, "-v"])
"""

with open(f'{path}/tests/test_foodquest_pnl.py', 'w') as f:
    f.write(test_code)
    
print("✓ Created test_foodquest_pnl.py")

In [0]:
# Run the tests
%sh
python -m pytest /Workspace/Users/abhishekkumar.singh@techygeekhub.com/tests/test_foodquest_pnl.py -v --tb=short

In [0]:
from pyspark.sql.functions import *
import re, time
from pyspark.sql.window import Window

def to_snake_case(name):
    return re.sub(r'[\s\-]+', '_', name).lower()

def to_snake_case_df(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, to_snake_case(col_name))
    return df


def enrich_df(df):

    df_coa_master = spark.read.table("fq_dev_catalog.silver.dim_coa_master")
    df_location_master = spark.read.table("fq_dev_catalog.silver.dim_location_master")

    # Step 1: Join both master tables upfront
    df_all_masters = df.join(
        df_coa_master, 
        df_coa_master["account_number"].cast("string") == df["accountNumber"], 
        'inner'
    ).join(
        df_location_master,
        col("location") == df_location_master.netsuite_location_name,
        'left'
    )

    # Step 2: Create detail rows - CORRECTED ORDER
    df_detail = df_all_masters.select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        col("account_name"),            # 4
        col("name"),                    # 5
        col("accoun_type"),             # 6
        col("majour_group"),            # 7
        col("group"),                   # 8
        col("sub_group"),               # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Detail").alias("Detail/Total")  # 21
    )

    # Step 3: Sub group totals - SAME ORDER
    df_subgroup_totals = df_all_masters.groupBy(
        "location", "month", "year", "majour_group", "group", "sub_group",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        sum("amount").alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        concat(lit("Total "), col("sub_group")).alias("account_name"),  # 4
        concat(lit("Total "), col("sub_group")).alias("name"),          # 5
        lit(None).cast("string").alias("accoun_type"),  # 6
        lit(None).alias("majour_group"),            # 7
        lit(None).alias("group"),                   # 8
        lit(None).alias("sub_group"),               # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Total").alias("Detail/Total")  # 21
    )

    # Group totals - SAME ORDER
    df_group_totals = df_all_masters.groupBy(
        "location", "month", "year", "majour_group", "group",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        sum("amount").alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        concat(lit("Total "), col("group")).alias("account_name"),      # 4
        concat(lit("Total "), col("group")).alias("name"),              # 5
        lit(None).cast("string").alias("accoun_type"),  # 6
        lit(None).alias("majour_group"),            # 7
        lit(None).alias("group"),                   # 8
        lit(None).alias("sub_group"),  # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Total").alias("Detail/Total")  # 21
    )

    # Major group totals - SAME ORDER
    df_majour_group_totals = df_all_masters.groupBy(
        "location", "month", "year", "majour_group",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        sum("amount").alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        concat(lit("Total "), col("majour_group")).alias("account_name"),  # 4
        concat(lit("Total "), col("majour_group")).alias("name"),          # 5
        lit(None).cast("string").alias("accoun_type"),  # 6
        lit(None).alias("majour_group"),            # 7
        lit(None).alias("group"),      # 8
        lit(None).alias("sub_group"),  # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Grand Total").alias("Detail/Total")  # 21
    )

    # Gross Profit - SAME ORDER
    df_gross_profit = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        (abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
     abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0)))).alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        lit("Gross Profit").alias("account_name"),  # 4
        lit("Gross Profit").alias("name"),          # 5
        lit(None).cast("string").alias("accoun_type"),  # 6
        lit(None).alias("majour_group"),  # 7
        lit(None).alias("group"),      # 8
        lit(None).alias("sub_group"),  # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Grand Total").alias("Detail/Total")  # 21
    )
    
    # Operating Profit - SAME ORDER
    df_operating_profit = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        (abs((sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
          abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0)))) -
         abs(sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0)))).alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        lit("Operating Profit").alias("account_name"),  # 4
        lit("Operating Profit").alias("name"),          # 5
        lit(None).alias("accoun_type"),  # 6
        lit(None).alias("majour_group"),  # 7
        lit(None).alias("group"),      # 8
        lit(None).alias("sub_group"),  # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Grand Total").alias("Detail/Total")  # 21
    )

    # EBITDA - SAME ORDER
    df_ebitda = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        (((abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
           abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0)))) -
          abs(sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0)))) +
         abs(sum(when(col("majour_group") == "Depreciation & Amortization", col("amount")).otherwise(0)))).alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        lit("EBITDA").alias("account_name"),            # 4
        lit("EBITDA").alias("name"),                    # 5
        lit(None).cast("string").alias("accoun_type"),  # 6
        lit(None).alias("majour_group"),            # 7
        lit(None).alias("group"),      # 8
        lit(None).alias("sub_group"),  # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Grand Total").alias("Detail/Total")  # 21
    )

    # Net Profit - SAME ORDER
    df_net_profit = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        (((abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
           abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0)))) -
          abs(sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0)))) -
         abs(sum(when(col("majour_group") == "Finance Costs", col("amount")).otherwise(0))) -
         abs(sum(when(col("majour_group") == "Tax", col("amount")).otherwise(0)))).alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        lit("Net Profit/(Loss)").alias("account_name"),   # 4
        lit("Net Profit/(Loss)").alias("name"),           # 5
        lit(None).alias("accoun_type"),      # 6
        lit(None).alias("majour_group"),   # 7
        lit(None).alias("group"),      # 8
        lit(None).alias("sub_group"),  # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Grand Total").alias("Detail/Total")  # 21
    )

    # Step 4: Union all dataframes
    df_combined = (df_detail
        .unionAll(df_subgroup_totals)
        .unionAll(df_group_totals)
        .unionAll(df_majour_group_totals)
        .unionAll(df_gross_profit)
        .unionAll(df_operating_profit)
        .unionAll(df_ebitda)
        .unionAll(df_net_profit))

    # Step 5: Extract year and prepare for PY calculations
    df_current = df_combined.withColumn("year", year(col("year")))

    # Step 6: Calculate Previous Year Sales (PY) - Self join
    df_py = df_current.alias("py").select(
        (col("year") + 1).alias("year_join"),
        col("location").alias("py_location"),
        col("account_name").alias("py_account_name"),
        col("amount").alias("py_amount"),
        col("month").alias("py_month")
    )

    df_with_py = df_current.alias("curr").join(
        df_py,
        (col("curr.year") == col("year_join")) &  
        (col("curr.location") == col("py_location")) &  
        (col("curr.account_name") == col("py_account_name")) &  
        (col("curr.month") == col("py_month")),  
        "left"
    ).select(
        col("curr.*"),
        coalesce(col("py_amount"), lit(0.0)).alias("py_amount")
    )

    # Step 7: Calculate Net Sales with window functions
    window_location = Window.partitionBy("location", "year", "month")
    window_brand = Window.partitionBy("brand_id", "year", "month")
    window_company = Window.partitionBy("company_id", "year", "month")

    df_with_calculations = df_with_py.withColumn(
        "actual_net_sales",
        sum(when(
            (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_location)
    ).withColumn(
        "py_net_sales",
        sum(when(
            (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_location)
    ).withColumn(
        "brand_act_net_sales",
        sum(when(
            (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_brand)
    ).withColumn(
        "brand_py_net_sales",
        sum(when(
            (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_brand)
    ).withColumn(
        "company_act_net_sales",
        sum(when(
            (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_company)
    ).withColumn(
        "company_py_net_sales",
        sum(when(
            (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_company)
    )

    # Step 8: Create final output with all required columns
    df_final = df_with_calculations.select(
        col("account_name"),
        col("name"),
        col("accoun_type"),
        col("majour_group"),
        col("group"),
        col("sub_group"),
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("parent_company"),
        col("country_code"),
        col("zone"),
        col("store_type"),
        col("city"),
        col("Detail/Total"),
        col("amount"),
        col("year"),
        col("month"),
        col("actual_net_sales"),
        col("py_net_sales"),
        col("brand_act_net_sales"),
        col("brand_py_net_sales"),
        col("company_act_net_sales"),
        col("company_py_net_sales")
    )

    df_final = to_snake_case_df(df_final)

    df_sort = df_coa_master.select("account_name", "sort_order", "calculation_type")
    df_final_sort = df_final.join(
            df_sort, 
            df_sort["account_name"].cast("string") == df_final.account_name,
            'inner'
        ).drop(df_sort["account_name"]).select(
            col("account_name"),
            col("name"),
            col("majour_group"),
            col("group"),
            col("sub_group"),
            # lit(None).cast("string"),
            col("accoun_type"),
            col("netsuite_location_name"),
            col("type"),
            col("location_id"),
            col("brand_id"),
            col("company_id"),
            # lit(None).cast("string").alias("Cluster"),
            col("store_type"),
            col("parent_company"),
            col("country_code"),
            col('zone'),
            col("city"),
            col("amount"),
            col("year"),
            col("month"),
            col("actual_net_sales"),
            col("py_net_sales"),
            col("brand_act_net_sales"),
            col("brand_py_net_sales"),
            col("company_act_net_sales"),
            col("company_py_net_sales"),
            col("detail/total"),
            col('sort_order'),
            col('calculation_type')
        )

    fact_financial_pnl = df_final_sort.orderBy("parent_company", "company_id", "brand_id", "netsuite_location_name", 'year', 'month', 'sort_order')
    return fact_financial_pnl

In [0]:

df = spark.read.option('multiline', False).format('json').load(f'{staging}/FoodQuest/Netsuite/Wastage/ALBAIK/2025/May/wastage_20250501_20250531.json')
exploded_df = (
            df.select(
                explode('results').alias('result')
            ).select('result.*')
        )
display(exploded_df)



In [0]:
from foodquest_pnl import *

# Now use refactored enrich_df function
result = enrich_df(exploded_df)
result.display()

In [0]:
df_final = enrich_df(exploded_df)
df_final.display()

In [0]:
%sql
CREATE EXTERNAL TABLE IF NOT EXISTS fq_dev_catalog.silver.fact_financial_pnl (
  account_name STRING,
  account_number STRING,
  majour_group STRING,
  group_name STRING,
  sub_group STRING,
  alternate_group STRING,
  account_type STRING,
  location STRING,
  type STRING COMMENT 'HO/Store',
  location_code STRING,
  brand STRING,
  company STRING,
  cluster STRING,
  location_mode STRING COMMENT 'Mall/Drive thru/Stand alone',
  emirates STRING,
  detail_total_grand_total STRING,
  sum_order INT,
  actual_value DECIMAL(18,2),
  budget DECIMAL(18,2),
  previous_year_sales DECIMAL(18,2),
  year INT,
  month INT,
  forecast DECIMAL(18,2),
  calculation_type STRING,
  actual_net_sales DECIMAL(18,2),
  budget_net_sales DECIMAL(18,2),
  py_net_sales DECIMAL(18,2),
  brand_act_net_sales DECIMAL(18,2),
  brand_budget_net_sales DECIMAL(18,2),
  brand_py_net_sales DECIMAL(18,2),
  company_act_net_sales DECIMAL(18,2),
  company_budget_net_sales DECIMAL(18,2),
  company_py_net_sales DECIMAL(18,2)
)
USING DELTA
CLUSTER BY (year, month, company, brand)
LOCATION 'abfss://fq-dev-silver-container@fqadfstoragedev.dfs.core.windows.net/external/fact_financial_pnl' 
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
def merge_stream_fact_financial_pnl(df, i):
    try:
        exploded_df = (
            df.select(
                explode('results').alias('result')
            ).select('result.*')
        )
        
        fact_financial_pnl_upsert = enrich_json(exploded_df)
        fact_financial_pnl_upsert.createOrReplaceTempView("fact_financial_pnl_upsert_microbatch")
       
        df.sparkSession.sql("""
            MERGE INTO fq_dev_catalog.silver.fact_financial_pnl target
            USING (
                SELECT *
                FROM fact_financial_pnl_upsert_microbatch
            ) as source
            ON target.year = source.year
                AND target.month = source.month
                AND target.location = source.location
                AND target.account_name = source.account_name
                AND target.sum_order = source.sum_order
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        
        print(f"Successfully merged batch {i}")

    # df.sparkSession.sql("""
    #     MERGE INTO fq_dev_catalog.silver.fact_financial_pnl target
    #     USING (
    #         SELECT *
    #         FROM (
    #             SELECT *, 
    #                 ROW_NUMBER() OVER (
    #                     PARTITION BY year, month, location, account_name, sum_order
    #                     ORDER BY year DESC  -- or add a load_time column
    #                 ) as rank
    #             FROM fact_financial_pnl_upsert_microbatch
    #         )
    #         WHERE rank = 1
    #     ) as source
    #     ON target.year = source.year
    #         AND target.month = source.month
    #         AND target.location = source.location
    #         AND target.account_name = source.account_name
    #         AND target.sum_order = source.sum_order
    #     WHEN MATCHED THEN UPDATE SET *
    #     WHEN NOT MATCHED THEN INSERT *
    # """)
    except Exception as e:
        print(f"Error in merge_stream: {e}")
        raise e

(spark.readStream
    # .option("schemaTrackingLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpointing_fact_financial_pnl/schema_fact_financial_pnl')
    .table("fq_dev_catalog.bronze.gl_report")
    .writeStream
    .foreachBatch(merge_stream_fact_financial_pnl)
    .option("mergeSchema", "true")
    .option('skipChangeCommits', "true")
    .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_silver_fact_financial_pnl3')  
    .trigger(availableNow=True)
    .start()
).awaitTermination()

time.sleep(20)

In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows
FROM fq_dev_catalog.silver.fact_financial_pnl;

In [0]:
%sql
select * from fq_dev_catalog.silver.fact_financial_pnl limit 1

In [0]:
for query in spark.streams.active:
    query.stop()